
# Pesquisa Informada — Greedy Best-First Search

Neste notebook vamos testar o algoritmo **Greedy Best-First Search** com duas heurísticas, usando os datasets `data_tiny` e `data_small`.

**Objetivo:** recolher os produtos nos produtores, passar pelo armazém e entregar as encomendas completas, respeitando capacidades, horários e o horizonte de planeamento.

O custo é a **distância total em quilómetros**, incluindo o regresso dos veículos ao armazém.

O Greedy procura soluções válidas, mas não garante encontrar a rota mais curta possível.



## 1. Preparação

Primeiro identificamos a pasta principal do projeto e importamos as funções necessárias.

Assim, o notebook funciona tanto a partir da raiz do projeto como da pasta `notebooks/`.


In [1]:

from pathlib import Path
import sys
import time
import pandas as pd

PROJECT = Path.cwd()

if not (PROJECT / "src").is_dir():
    PROJECT = PROJECT.parent

if not (PROJECT / "src").is_dir():
    raise FileNotFoundError("Pasta src/ não encontrada.")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from src.model import Problem, check_solution, simulate
from src.informed import (
    greedy_search,
    heuristic01,
    heuristic02
)

print("Projeto:", PROJECT)


Projeto: /Users/davidfaria/Documents/IPCA 26_27/IA/IA2026_TP_G05



## 2. Carregar os datasets

Vamos utilizar dois conjuntos de dados:

- **data_tiny:** 12 encomendas.
- **data_small:** 25 encomendas.

A função `validate()` verifica possíveis problemas nos dados de entrada.

Uma lista sem avisos não garante que exista uma solução, mas confirma que não foram detetadas as inconsistências verificadas por essa função.


In [2]:

problems = {}

for name in ("data_tiny", "data_small"):

    problem = Problem.from_csv(PROJECT / name)
    problems[name] = problem

    issues = problem.validate()

    print(f"\nDataset: {name}")
    print("Encomendas:", len(problem.orders))
    print("Itens:", len(problem.requests))
    print("Veículos:", len(problem.vehicles))
    print("Avisos:", issues if issues else "Nenhum")



Dataset: data_tiny
Encomendas: 12
Itens: 40
Veículos: 2
Avisos: Nenhum

Dataset: data_small
Encomendas: 25
Itens: 80
Veículos: 2
Avisos: Nenhum



## 3. Heurísticas utilizadas

O Greedy Best-First Search escolhe primeiro o estado com menor valor heurístico, usando uma fila de prioridade (`heapq`).

### H1 — Operações pendentes

Conta o trabalho que ainda falta realizar, considerando recolhas, descargas, carregamentos e entregas.

Quanto menos operações faltarem, mais interessante é o estado.

### H2 — Operações e distância

Utiliza primeiro as operações pendentes e, em caso de empate, considera uma estimativa dos quilómetros restantes.

A H2 devolve um tuplo `(operações, distância_estimada)`, que é comparado por ordem.

**Nota:** nenhuma destas heurísticas garante que o Greedy encontre uma solução ótima.



## 4. Executar o Greedy Best-First Search

Vamos executar as heurísticas H1 e H2 nos dois datasets.

Para cada execução, registamos:

- Se encontrou uma solução válida.
- A distância total em quilómetros.
- O número de nós expandidos.
- O tempo de execução.

Todas as soluções encontradas são verificadas com `check_solution()`.

O limite de 5000 expansões aplica-se a cada tentativa. O algoritmo pode repetir a pesquisa sem o filtro temporal caso a primeira tentativa falhe.


In [3]:

heuristics = {
    "H1": heuristic01,
    "H2": heuristic02
}

results = []
solutions = {}

for dataset, problem in problems.items():

    for name, h in heuristics.items():

        start = time.perf_counter()

        solution, expanded = greedy_search(
            problem,
            problem.initial_state(),
            h,
            max_expansions=5000
        )

        elapsed = time.perf_counter() - start

        if solution is not None:
            valid, km, message = check_solution(
                problem, solution
            )
        else:
            valid = False
            km = None
            message = "Sem solução no limite definido"

        solutions[(dataset, name)] = (
            solution if valid else None
        )

        results.append({
            "Dataset": dataset,
            "Heurística": name,
            "Solução válida": valid,
            "Distância (km)": km,
            "Nós expandidos": expanded,
            "Tempo (s)": round(elapsed, 2),
            "Mensagem": message
        })

        print(f"\n{dataset} — {name}")
        print("Solução válida:", valid)
        print("Distância:", km, "km")
        print("Nós expandidos:", expanded)
        print(f"Tempo: {elapsed:.2f} s")



data_tiny — H1
Solução válida: True
Distância: 589.19 km
Nós expandidos: 555
Tempo: 0.05 s

data_tiny — H2
Solução válida: True
Distância: 735.4 km
Nós expandidos: 60
Tempo: 0.02 s

data_small — H1
Solução válida: True
Distância: 974.22 km
Nós expandidos: 70
Tempo: 0.11 s

data_small — H2
Solução válida: True
Distância: 1134.67 km
Nós expandidos: 393
Tempo: 0.18 s



## 5. Comparação dos resultados


Uma distância menor representa uma solução melhor entre as rotas encontradas.

O número de nós expandidos permite observar o esforço de pesquisa. O tempo de execução também depende do computador utilizado.

No final, identificamos a melhor solução encontrada em cada dataset.


In [4]:

df = pd.DataFrame(results)

display(df)

for dataset in problems:

    valid_rows = df[
        (df["Dataset"] == dataset)
        & (df["Solução válida"])
    ]

    if valid_rows.empty:
        print(f"{dataset}: sem solução válida")
        continue

    best = valid_rows.loc[
        valid_rows["Distância (km)"].idxmin()
    ]

    print(
        f"{dataset}: melhor resultado = "
        f"{best['Heurística']} "
        f"({best['Distância (km)']:.2f} km)"
    )


,Dataset,Heurística,Solução válida,Distância (km),Nós expandidos,Tempo (s),Mensagem
0,data_tiny,H1,True,589.19,555,0.05,Solução válida
1,data_tiny,H2,True,735.40,60,0.02,Solução válida
2,data_small,H1,True,974.22,70,0.11,Solução válida
3,data_small,H2,True,1134.67,393,0.18,Solução válida


data_tiny: melhor resultado = H1 (589.19 km)
data_small: melhor resultado = H1 (974.22 km)



## 6. Visualizar as rotas

Vamos apresentar a melhor solução encontrada para cada dataset.

A função `print_solution()` mostra as operações realizadas pelos veículos, incluindo:

- Recolha dos produtos (`pickup`).
- Descarga no armazém (`unload`).
- Carregamento das encomendas (`load`).
- Entrega aos clientes (`deliver`).
- Regresso ao armazém (`return`).

São também apresentados os horários e as distâncias percorridas.


In [5]:
for dataset, problem in problems.items():

    valid_rows = df[
        (df["Dataset"] == dataset)
        & (df["Solução válida"])
    ]

    if valid_rows.empty:
        print(f"{dataset}: sem rota")
        continue

    best = valid_rows.loc[
        valid_rows["Distância (km)"].idxmin()
    ]

    name = best["Heurística"]
    solution = solutions[(dataset, name)]

    result = simulate(problem, solution)

    print("\n" + "=" * 60)
    print(f"{dataset} — Melhor heurística: {name}")
    print(f"Distância total: {result.distance:.2f} km")
    print(f"Validação: {result.message}")
    print("=" * 60)

    for vehicle in problem.vehicles:
        print(f"\nVeículo {vehicle.id}")

        for event in result.events:
            v, action, place, ids, arrival, start, finish, km = event

            if v != vehicle.id:
                continue

            print(
                f"  {action:8s} | "
                f"{problem.name(place):22s} | "
                f"{start:%d/%m %H:%M} | "
                f"{km:.2f} km"
            )


data_tiny — Melhor heurística: H1
Distância total: 589.19 km
Validação: Solução válida

Veículo 1
  pickup   | Horta Morais           | 30/10 08:00 | 29.06 km
  pickup   | Campos Leite           | 30/10 09:00 | 26.05 km
  unload   | Centro de Recolha 1    | 30/10 10:00 | 37.66 km
  load     | Centro de Recolha 1    | 30/10 10:35 | 0.00 km
  deliver  | Loja 1                 | 30/10 11:15 | 18.79 km
  deliver  | Casa 3                 | 30/10 11:34 | 6.66 km
  deliver  | Casa 4                 | 30/10 12:18 | 25.25 km
  deliver  | Casa 4                 | 31/10 07:15 | 0.00 km
  deliver  | Loja 5                 | 31/10 08:15 | 25.25 km
  pickup   | Hortas Valente         | 31/10 09:00 | 14.86 km
  pickup   | Campos Leite           | 31/10 09:41 | 23.59 km
  unload   | Centro de Recolha 1    | 31/10 10:41 | 37.66 km
  load     | Centro de Recolha 1    | 31/10 10:56 | 0.00 km
  deliver  | Casa 2                 | 31/10 11:59 | 36.12 km
  deliver  | Loja 5                 | 31/10 13:25 |


## 7. Verificar as encomendas entregues

Uma solução só está completa quando todos os itens foram entregues.

Vamos confirmar:

- Quantas encomendas foram entregues.
- Quantos itens foram entregues.
- Se o validador aprova a solução.

O `check_solution()` verifica também a sequência obrigatória das quatro operações e as restrições implementadas no modelo.


In [6]:

for (dataset, name), solution in solutions.items():

    if solution is None:
        print(f"{dataset} — {name}: sem solução válida")
        continue

    problem = problems[dataset]

    valid, km, message = check_solution(
        problem, solution
    )

    delivered_items = {
        rid
        for route in solution.values()
        for action, rid in route
        if action == "deliver"
    }

    delivered_orders = {
        problem.request[rid].order_id
        for rid in delivered_items
    }

    all_items = set(problem.request)

    print(
        f"{dataset} — {name}: "
        f"{len(delivered_orders)}/{len(problem.orders)} encomendas, "
        f"{len(delivered_items)}/{len(all_items)} itens, "
        f"válida={valid}, {km:.2f} km"
    )

    assert valid and delivered_items == all_items


data_tiny — H1: 12/12 encomendas, 40/40 itens, válida=True, 589.19 km
data_tiny — H2: 12/12 encomendas, 40/40 itens, válida=True, 735.40 km
data_small — H1: 25/25 encomendas, 80/80 itens, válida=True, 974.22 km
data_small — H2: 25/25 encomendas, 80/80 itens, válida=True, 1134.67 km



## 8. Teste de uma solução inválida

Para verificar o comportamento do validador, vamos criar uma solução sem operações.

Como nenhuma encomenda é entregue, esta solução deve ser rejeitada.

Este teste confirma que o validador não aceita uma rota vazia como solução completa.


In [7]:

problem = problems["data_tiny"]

empty_routes = {
    vehicle.id: []
    for vehicle in problem.vehicles
}

valid, km, message = check_solution(
    problem,
    empty_routes
)

print("Rota vazia válida?", valid)
print("Motivo:", message)

assert not valid


Rota vazia válida? False
Motivo: Encomendas por entregar



## 9. Conclusão

Neste notebook foi aplicado o algoritmo **Greedy Best-First Search** ao problema de recolha e entrega de produtos locais.

Foram comparadas duas heurísticas:

- **H1:** operações pendentes.
- **H2:** operações pendentes com desempate pela distância estimada.

A avaliação considera a validade das rotas, a distância total, o número de nós expandidos e o tempo de execução.

As soluções são verificadas pelo modelo, incluindo as capacidades dos veículos, as janelas temporais, a passagem pelo armazém e a entrega completa das encomendas.

**Limitação:** o Greedy não garante encontrar a solução com menor distância possível.

Além disso, o validador faz parte do mesmo projeto que gera os movimentos, pelo que testes independentes podem reforçar a confiança nos resultados.

**Trabalho futuro:** comparar o Greedy com o algoritmo A*, desenvolvido separadamente.
